In [4]:
import re
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── Run configuration ──
RUN_ID = 'manual_cal026'
RUN_PATH = Path.cwd() / f'run_{RUN_ID}'
TRANS_FILE = RUN_PATH / f'pflotran-mcp19_{RUN_ID}.h5'
IN_FILE = RUN_PATH / f'pflotran-mcp19_{RUN_ID}.in'

# ── Grid constants ──
NX, NY, NZ = 1, 122, 26
DX, DY, DZ = 1.0, 0.5, 0.1  # meters
CELL_VOL = DX * DY * DZ       # 0.05 m³

startdate = np.datetime64('2019-04-21')

print(f'Run: {RUN_ID}')
print(f'H5:  {TRANS_FILE.name} — exists: {TRANS_FILE.exists()}')
print(f'IN:  {IN_FILE.name} — exists: {IN_FILE.exists()}')

Run: manual_cal026
H5:  pflotran-mcp19_manual_cal026.h5 — exists: True
IN:  pflotran-mcp19_manual_cal026.in — exists: True


In [5]:
def extract_aerobic_params(in_file):
    """Parse MICROBIAL_REACTION block for aerobic respiration Monod parameters."""
    text = Path(in_file).read_text()

    # Match MICROBIAL_REACTION block using indentation-aware closing.
    # The backreference \1 ensures we stop at the `/` that matches the
    # indentation of MICROBIAL_REACTION, not the inner MONOD closers.
    pattern = re.compile(
        r'^(\s*)MICROBIAL_REACTION\s*\n(.*?)\n\1/',
        re.DOTALL | re.MULTILINE
    )
    match = pattern.search(text)
    if not match:
        raise ValueError('Could not find MICROBIAL_REACTION block')
    block = match.group(2)

    # Rate constant
    rc = re.search(r'RATE_CONSTANT\s+([\d.eEdD+-]+)', block)
    rate_constant = float(rc.group(1).replace('d', 'e').replace('D', 'E'))

    # Parse each MONOD sub-block (same indentation-aware approach)
    monod_iter = re.finditer(
        r'^(\s*)MONOD\s*\n(.*?)\n\1/',
        block, re.DOTALL | re.MULTILINE
    )

    params = {'rate_constant': rate_constant}
    for m in monod_iter:
        mb = m.group(2)
        species = re.search(r'SPECIES_NAME\s+(\S+)', mb).group(1)
        ks = float(re.search(r'HALF_SATURATION_CONSTANT\s+([\d.eEdD+-]+)', mb)
                   .group(1).replace('d', 'e').replace('D', 'E'))
        tc = re.search(r'THRESHOLD_CONCENTRATION\s+([\d.eEdD+-]+)', mb)
        thresh = float(tc.group(1).replace('d', 'e').replace('D', 'E')) if tc else 0.0

        if 'O2' in species:
            params['km_o2'] = ks
            params['thresh_o2'] = thresh
        elif 'SOC' in species:
            params['km_soc'] = ks
            params['thresh_soc'] = thresh

    print('Aerobic respiration parameters:')
    for k, v in params.items():
        print(f'  {k:20s} = {v:.2e}')
    return params

aer_params = extract_aerobic_params(IN_FILE)

Aerobic respiration parameters:
  rate_constant        = 5.00e-10
  km_o2                = 1.00e-06
  thresh_o2            = 1.00e-08
  km_soc               = 1.00e-06
  thresh_soc           = 1.00e-08


In [ ]:
def monod(conc, km, thresh):
    """Monod term with threshold: max(0, C-thresh) / (Km + max(0, C-thresh))."""
    eff = np.maximum(0.0, conc - thresh)
    return eff / (km + eff)


# JB sandbox rate dataset names → short labels
JB_DATASETS = {
    'JB Fh Acetate Sandbox Rate': 'Fe(III) Red. (Fh)',
    'JB Gt Acetate Sandbox Rate': 'Fe(III) Red. (Gt)',
    'JB Sulfate Acetate Sandbox Rate': 'Sulfate Reduction',
    'JB Nitrate Acetate Sandbox Rate': 'Denitrification',
}
# All JB reactions: 1 mol Ac- → 2 mol HCO3-
JB_DIC_STOICH = 2.0


def compute_dic_rates(h5_path, params):
    """Extract DIC production rates from each timestep in the transient HDF5.

    Returns dict with:
      times_h  : array of times in hours
      dates    : array of datetime64 dates
      root_resp: total Root_Respiration rate [mol/s]
      calcite  : total Calcite rate [mol/s]
      aerobic  : total aerobic respiration rate [mol/s] (computed from Monod)
      jb_*     : total JB anaerobic rates [mol DIC/s] (if present in HDF5)
    """
    k = params['rate_constant']
    km_o2, thresh_o2 = params['km_o2'], params['thresh_o2']
    km_soc, thresh_soc = params['km_soc'], params['thresh_soc']

    times_h = []
    root_resp = []
    calcite = []
    aerobic = []
    jb_rates = {name: [] for name in JB_DATASETS}

    with h5py.File(h5_path, 'r') as f:
        time_groups = sorted(
            [g for g in f.keys() if g.startswith('Time')],
            key=lambda g: float(g.split()[1])
        )
        print(f'Found {len(time_groups)} timesteps')

        # Detect which JB datasets are present (check first time group)
        sample_keys = set(f[time_groups[0]].keys())
        jb_found = {}
        for ds_name in JB_DATASETS:
            # Match with or without unit suffix
            matches = [k for k in sample_keys if k.startswith(ds_name)]
            if matches:
                jb_found[ds_name] = matches[0]
                print(f'  Found: {matches[0]}')
        if not jb_found:
            print('  No JB sandbox rate datasets found — re-run with updated OUTPUT block')

        for gname in time_groups:
            grp = f[gname]
            t_h = float(gname.split()[1])
            times_h.append(t_h)

            # Root respiration [mol/m³/s] → total mol/s
            rr = grp['Root_Respiration_Rate [mol_m^3_sec]'][:]
            root_resp.append(np.sum(rr) * CELL_VOL)

            # Calcite [mol/m³/s] → total mol/s
            cr = grp['Calcite_Rate [mol_m^3_sec]'][:]
            calcite.append(np.sum(cr) * CELL_VOL)

            # Aerobic: Monod kinetics
            o2 = grp['Free_O2(aq) [M]'][:]
            soc = grp['Free_SOC(aq) [M]'][:]
            rate_field = k * monod(o2, km_o2, thresh_o2) * monod(soc, km_soc, thresh_soc)
            aerobic.append(np.sum(rate_field) * CELL_VOL)

            # JB anaerobic rates [mol Ac-/s per cell] → mol DIC/s
            for ds_name in JB_DATASETS:
                if ds_name in jb_found:
                    jb = grp[jb_found[ds_name]][:]
                    jb_rates[ds_name].append(np.sum(jb) * JB_DIC_STOICH)
                else:
                    jb_rates[ds_name].append(0.0)

    times_h = np.array(times_h)
    dates = startdate + (times_h * 3600).astype('timedelta64[s]')

    result = {
        'times_h': times_h,
        'dates': dates,
        'root_resp': np.array(root_resp),
        'calcite': np.array(calcite),
        'aerobic': np.array(aerobic),
    }
    for ds_name in JB_DATASETS:
        result[ds_name] = np.array(jb_rates[ds_name])

    return result

rates = compute_dic_rates(TRANS_FILE, aer_params)

In [ ]:
# Convert mol/s → µmol/s
root = rates['root_resp'] * 1e6
aerob = rates['aerobic'] * 1e6
calc = rates['calcite'] * 1e6
dates = rates['dates']

# Anaerobic JB rates (already in mol DIC/s from compute_dic_rates)
jb_colors = {
    'JB Nitrate Acetate Sandbox Rate': '#9C27B0',
    'JB Fh Acetate Sandbox Rate':     '#4CAF50',
    'JB Gt Acetate Sandbox Rate':     '#009688',
    'JB Sulfate Acetate Sandbox Rate': '#795548',
}
jb_arrays = {}
for ds_name in JB_DATASETS:
    jb_arrays[ds_name] = rates[ds_name] * 1e6

# Split calcite into dissolution (positive) and precipitation (negative)
calc_diss = np.maximum(calc, 0)
calc_prec = np.minimum(calc, 0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

# ── Top panel: stacked area + calcite net ──
cumul = np.zeros_like(root)

ax1.fill_between(dates, cumul, cumul + root, label='Root Respiration',
                 alpha=0.8, color='#8B4513')
cumul = cumul + root

ax1.fill_between(dates, cumul, cumul + aerob, label='Aerobic Respiration',
                 alpha=0.8, color='#2196F3')
cumul = cumul + aerob

# Stack anaerobic JB rates
for ds_name, label in JB_DATASETS.items():
    arr = jb_arrays[ds_name]
    if np.any(arr != 0):
        ax1.fill_between(dates, cumul, cumul + arr, label=label,
                         alpha=0.8, color=jb_colors[ds_name])
        cumul = cumul + arr

ax1.fill_between(dates, cumul, cumul + calc_diss,
                 label='Calcite Dissolution', alpha=0.8, color='#FF9800')
cumul_with_calc = cumul + calc_diss

ax1.plot(dates, calc, color='#E91E63', lw=1.2, ls='--', label='Net Calcite')

if np.any(calc_prec < 0):
    ax1.fill_between(dates, 0, calc_prec, alpha=0.4, color='#E91E63',
                     label='Calcite Precipitation')

ax1.set_ylabel(r'DIC Production [$\mu$mol C s$^{-1}$]')
ax1.legend(loc='upper right', fontsize=8)
ax1.set_title(f'DIC Sources — {RUN_ID}')
sns.despine(ax=ax1)

# ── Bottom panel: fractional contribution ──
total_pos = cumul_with_calc.copy()
total_pos_safe = np.where(total_pos > 0, total_pos, np.nan)
cum_frac = np.zeros_like(root)

frac_items = [('Root Respiration', root, '#8B4513'),
              ('Aerobic Respiration', aerob, '#2196F3')]
for ds_name, label in JB_DATASETS.items():
    arr = jb_arrays[ds_name]
    if np.any(arr != 0):
        frac_items.append((label, arr, jb_colors[ds_name]))
frac_items.append(('Calcite Dissolution', calc_diss, '#FF9800'))

for _, arr, color in frac_items:
    frac = arr / total_pos_safe * 100
    ax2.fill_between(dates, cum_frac, cum_frac + frac, alpha=0.8, color=color)
    cum_frac = cum_frac + frac

ax2.set_ylabel('Contribution [%]')
ax2.set_ylim(0, 100)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax2.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
fig.autofmt_xdate(rotation=30)
sns.despine(ax=ax2)

plt.tight_layout()
fig.savefig(RUN_PATH / f'{RUN_ID}_dic_production.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics (µmol C / s)
labels = ['Root Respiration', 'Aerobic Respiration']
arrays = [root, aerob]

for ds_name, label in JB_DATASETS.items():
    arr = jb_arrays[ds_name]
    if np.any(arr != 0):
        labels.append(label)
        arrays.append(arr)

labels.append('Calcite (net)')
arrays.append(calc)

total_pos_mean = np.nanmean(
    root + aerob + sum(jb_arrays.values()) + calc_diss
)

print(f'{"Source":>25s}   {"Mean":>10s}   {"Min":>10s}   {"Max":>10s}   {"% of total":>10s}')
print('-' * 75)

for lbl, arr in zip(labels, arrays):
    mn = np.nanmean(arr)
    pct = mn / total_pos_mean * 100 if total_pos_mean > 0 else 0
    print(f'{lbl:>25s}   {mn:10.4f}   {np.nanmin(arr):10.4f}   {np.nanmax(arr):10.4f}   {pct:9.1f}%')